# Vehicle-to-grid energy-cost savings

Run all cells from the repository root using the Python environment in `requirements.txt`. The notebook uses the 67 numerical fleet inputs distributed here. Each run creates its own output folder. The studied tariff pays no credit for exports and includes no peak demand fee.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import hashlib
import subprocess
import sys

root = Path.cwd()
if not (root / "inputs_sha256.csv").is_file():
    raise FileNotFoundError("Open this notebook from the repository root.")
run = root / "runs" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
results = run / "results"
figures = run / "figures"
results.mkdir(parents=True)
figures.mkdir()

def execute(script, *args):
    subprocess.run([sys.executable, str(root / script), *map(str, args)], cwd=root, check=True)

with (root / "inputs_sha256.csv").open(encoding="utf-8", newline="") as stream:
    manifest = list(csv.DictReader(stream))
for record in manifest:
    path = root / record["relative_path"]
    if hashlib.sha256(path.read_bytes()).hexdigest() != record["sha256"]:
        raise ValueError(f"The input digest differs for {path.name}")
print(f"Verified {len(manifest)} inputs. Writing to {run}")

## Schedule the recorded fleets

Compare optimized charging only, the schedule assuming full discharge delivery, and the schedule covering one withdrawn discharge commitment. Then repeat the protected comparison under both seasonal tariffs and on all 31 dates of July.

In [ ]:
execute("run_study.py", "all", "--out", results)

## Test the threshold and the effect of the import limit

The threshold sweep solves 540 fleet/budget pairs and directly enumerates 132 of them as an additional check. The common-limit study reoptimizes both policies at the same connection rating.

In [ ]:
execute("src/validate_threshold.py", "--fleets", root / "inputs/study", "--out", results)
execute("run_cap_sensitivity.py", "--out", results)

## Check battery modes and the payment sensitivity

The binary-mode experiment runs one independent verification for each of the 36 fleets. Runtime depends on the machine. The final calculation evaluates a payment per kWh of scheduled discharge.

In [ ]:
execute("mode_replication/mode_check.py", "--fleets", root / "inputs/study",
        "--published", results / "q_sweep.csv", "--out", results / "mode_checks.csv", "--limit", 20)
execute("mode_replication/surplus_check.py", "--out", results / "surplus_mode_check.csv")
execute("payment_replication/discharge_allowance.py", results, "--out", results / "discharge_allowance.csv")

## Draw the five figures and read the bills

The four empirical figures use the results from this run. The timetable figure is recomputed from the six-vehicle input. The contract diagram illustrates the operator's decision.

In [ ]:
execute("plots/plot_energy_results.py", "--results", results, "--output-dir", figures)
execute("plots/plot_contract.py", "--output", figures / "contract_meter.pdf")
with (results / "common_cap_summary.csv").open(encoding="utf-8", newline="") as stream:
    for row in csv.DictReader(stream):
        print(f"Cap {row['multiplier']:>9}: saving USD {float(row['total_saving_usd']):6.2f}"
              f" ({float(row['pct_total_charging_only']):.2f}%)")
with (results / "july_totals.csv").open(encoding="utf-8", newline="") as stream:
    july = next(csv.DictReader(stream))
print(f"July: saving USD {float(july['savings_usd']):.2f} ({float(july['savings_percent']):.2f}%)")
print(f"Figure PDFs: {figures}")